# Amazon Sentiment Analysis: Research Paper Replication (ML-Only)

This notebook is a cleaned, rerunnable implementation of the paper's **traditional ML** pipeline across:
- Appliances
- Groceries
- Clothing

Scope in this notebook:
- Full preprocessing pipeline from the paper (contractions, emoji, language filtering, date filtering, HTML stripping)
- LazyPredict screening on 20,000 samples/domain
- Final experiments on 100,000 samples/domain (stratified, natural imbalance preserved)
- Exactly 10 base models + StackFull + StackSHAP
- SHAP model-level ranking on stacking meta-features
- SHAP feature-level interpretation for **Bagging only**
- McNemar statistical validation
- Required paper-style plots and tables
- Extra analysis retained: learning curves and ROC/AUC

Deep learning / transformer models are intentionally excluded.


In [5]:
# Install required libraries (run once if needed)
!pip install -q datasets pandas numpy scikit-learn nltk spacy emoji langdetect beautifulsoup4 contractions shap lazypredict xgboost lightgbm statsmodels seaborn matplotlib

zsh:1: command not found: pip


## 1. Imports and Global Configuration

In [4]:
import warnings
warnings.filterwarnings('ignore')

import re
import random
from collections import OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import spacy
import emoji
import contractions

from bs4 import BeautifulSoup
from datasets import load_dataset
from langdetect import detect, DetectorFactory, LangDetectException

from lazypredict.Supervised import LazyClassifier

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    balanced_accuracy_score,
)
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import label_binarize

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    StackingClassifier,
)

import xgboost as xgb
import lightgbm as lgb

import shap
from statsmodels.stats.contingency_tables import mcnemar

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DetectorFactory.seed = RANDOM_STATE

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


ModuleNotFoundError: No module named 'seaborn'

## 2. Experiment Constants and Dataset URLs

In [ ]:
DOMAIN_URLS = {
    'Clothing': 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl',
    'Groceries': 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Grocery_and_Gourmet_Food.jsonl',
    'Appliances': 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Appliances.jsonl',
}

LAZY_SAMPLE_SIZE = 20_000
FINAL_SAMPLE_SIZE = 100_000
BUFFER_FACTOR = 1.5

MIN_YEAR = 2013
MAX_YEAR = 2023

NEGATION_WORDS = {'not', 'no', 'never'}

print('LazyPredict sample size/domain:', LAZY_SAMPLE_SIZE)
print('Final training sample size/domain:', FINAL_SAMPLE_SIZE)
print('Year filter:', MIN_YEAR, 'to', MAX_YEAR)


## 3. NLP Resource Setup

In [ ]:
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

try:
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    from spacy.cli import download
    download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

stop_words = set(stopwords.words('english')) - NEGATION_WORDS
print('spaCy model loaded; stop words prepared with negation retention.')


## 4. Preprocessing Functions (Paper-Aligned)

This section implements the required preprocessing:
- contraction expansion (`contractions.fix`)
- emoji conversion (`emoji.demojize`)
- language filtering (`langdetect.detect == 'en'`)
- date filtering (2013-2023)
- HTML stripping (BeautifulSoup)
- tokenization + lemmatization (spaCy)


In [ ]:
def map_rating_to_sentiment(rating):
    if pd.isna(rating):
        return None
    rating = float(rating)
    if rating <= 2:
        return 0
    if rating == 3:
        return 1
    return 2


def parse_review_year(timestamp_value):
    if pd.isna(timestamp_value):
        return None

    try:
        ts = float(timestamp_value)
        if ts > 1e12:
            dt = pd.to_datetime(ts, unit='ms', utc=True, errors='coerce')
        elif ts > 1e9:
            dt = pd.to_datetime(ts, unit='s', utc=True, errors='coerce')
        else:
            dt = pd.to_datetime(timestamp_value, utc=True, errors='coerce')
    except Exception:
        dt = pd.to_datetime(timestamp_value, utc=True, errors='coerce')

    if pd.isna(dt):
        return None
    return int(dt.year)


def is_english_text(text):
    if not isinstance(text, str):
        return False
    text = text.strip()
    if len(text) < 5:
        return False
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False
    except Exception:
        return False


def clean_text_single(text):
    # HTML stripping before regex cleanup
    text = BeautifulSoup(str(text), 'html.parser').get_text(' ')

    # Emoji to text and contraction expansion
    text = emoji.demojize(text, delimiters=(' ', ' '))
    text = contractions.fix(text)

    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s:]', ' ', text)
    text = re.sub(r':([a-z_]+):', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_texts_batch(texts, batch_size=64):
    cleaned = [clean_text_single(t) for t in texts]
    lemmatized = []

    for doc in nlp.pipe(cleaned, batch_size=batch_size):
        tokens = []
        for token in doc:
            tok = token.lemma_.strip()
            if not tok or len(tok) < 2:
                continue
            if tok in stop_words or tok in NEGATION_WORDS:
                if tok in NEGATION_WORDS:
                    tokens.append(tok)
                continue
            tokens.append(tok)
        lemmatized.append(' '.join(tokens))

    return lemmatized


def stratified_natural_sample(df, target_size, label_col='label', random_state=RANDOM_STATE):
    if len(df) <= target_size:
        return df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    proportions = df[label_col].value_counts(normalize=True).sort_index()
    target_counts = (proportions * target_size).astype(int)

    remainder = target_size - int(target_counts.sum())
    if remainder > 0:
        for cls in proportions.sort_values(ascending=False).index[:remainder]:
            target_counts.loc[cls] += 1

    parts = []
    for cls, cnt in target_counts.items():
        cls_df = df[df[label_col] == cls]
        take = min(int(cnt), len(cls_df))
        parts.append(cls_df.sample(n=take, random_state=random_state))

    sampled = pd.concat(parts)

    if len(sampled) < target_size:
        remaining = df.drop(sampled.index)
        extra = remaining.sample(n=target_size - len(sampled), random_state=random_state)
        sampled = pd.concat([sampled, extra])

    return sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)


## 5. Data Loading with Natural Imbalance Preservation

- Stream each domain from source JSONL
- Keep only valid English reviews in 2013-2023
- Apply full preprocessing pipeline
- Build a stratified sample that preserves observed natural class imbalance


In [ ]:
def build_domain_dataframe(data_url, domain_name, target_size=FINAL_SAMPLE_SIZE, buffer_factor=BUFFER_FACTOR):
    buffer_size = int(target_size * buffer_factor)
    stream = load_dataset('json', data_files=data_url, split='train', streaming=True)

    rows = []
    for row in stream:
        rating = row.get('rating', None)
        text = row.get('text', '')
        timestamp = row.get('timestamp', None)

        if pd.isna(rating) or not isinstance(text, str) or not text.strip():
            continue

        year = parse_review_year(timestamp)
        if year is None or year < MIN_YEAR or year > MAX_YEAR:
            continue

        if not is_english_text(text):
            continue

        label = map_rating_to_sentiment(rating)
        if label is None:
            continue

        rows.append({
            'text': text,
            'rating': int(round(float(rating))),
            'label': label,
            'year': year,
        })

        if len(rows) >= buffer_size:
            break

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No rows collected for {domain_name}.')

    # De-duplicate and remove missing values
    df = df.dropna(subset=['text', 'rating', 'label', 'year'])
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

    print(f'{domain_name}: rows before text cleaning = {len(df):,}')
    df['clean_text'] = clean_texts_batch(df['text'].tolist(), batch_size=64)
    df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

    sampled = stratified_natural_sample(df, target_size=target_size, label_col='label', random_state=RANDOM_STATE)

    if len(sampled) < target_size:
        print(f'Warning: {domain_name} has only {len(sampled):,} rows after filtering.')

    print(f'{domain_name}: final sampled rows = {len(sampled):,}')
    print(f"{domain_name}: class distribution =", sampled['label'].value_counts(normalize=True).sort_index().round(4).to_dict())
    return sampled.reset_index(drop=True)


## 6. Build Final Datasets (100,000 per Domain)

In [ ]:
domain_dfs = {}
for domain_name, data_url in DOMAIN_URLS.items():
    print('\\n' + '=' * 80)
    print(f'Building domain dataset: {domain_name}')
    print('=' * 80)
    domain_dfs[domain_name] = build_domain_dataframe(
        data_url=data_url,
        domain_name=domain_name,
        target_size=FINAL_SAMPLE_SIZE,
        buffer_factor=BUFFER_FACTOR,
    )

for domain_name, df in domain_dfs.items():
    print(f"{domain_name}: shape={df.shape}")


## 7. Required Figure Plots: Rating and Class Distribution (Figs. 2 and 4 Style)

In [ ]:
def plot_rating_distribution(df, domain_name):
    counts = df['rating'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=counts.index.astype(str), y=counts.values, palette='viridis')
    plt.title(f'Rating Distribution (1-5): {domain_name}')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.show()


def plot_class_distribution(df, domain_name):
    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    counts = df['label'].map(mapping).value_counts().reindex(['Negative', 'Neutral', 'Positive'])
    plt.figure(figsize=(8, 4))
    sns.barplot(x=counts.index, y=counts.values, palette='mako')
    plt.title(f'Class Distribution: {domain_name}')
    plt.xlabel('Sentiment Class')
    plt.ylabel('Count')
    plt.show()


for domain_name, df in domain_dfs.items():
    plot_rating_distribution(df, domain_name)
    plot_class_distribution(df, domain_name)


## 8. LazyPredict Screening on 20,000 Stratified Samples per Domain

In [ ]:
def run_lazypredict_screen(df, domain_name, sample_size=LAZY_SAMPLE_SIZE):
    print('\n' + '-' * 80)
    print(f'LazyPredict screening: {domain_name}')
    print('-' * 80)

    sample_df = stratified_natural_sample(df, target_size=sample_size, label_col='label', random_state=RANDOM_STATE)

    X_train, X_test, y_train, y_test = train_test_split(
        sample_df['clean_text'],
        sample_df['label'],
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=sample_df['label'],
    )

    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, min_df=5, sublinear_tf=True)
    X_train_vec = vec.fit_transform(X_train)
    X_test_vec = vec.transform(X_test)

    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models, _ = clf.fit(X_train_vec, X_test_vec, y_train, y_test)

    # Keep only internal ranking reference; downstream experiments use fixed paper model set.
    return models


lazy_results = {}
for domain_name, df in domain_dfs.items():
    lazy_results[domain_name] = run_lazypredict_screen(df, domain_name)


## 9. Final Model Set (Fixed, Paper-Aligned)

Exactly 10 models are used for all main experiments:
1. Logistic Regression (LR)
2. LinearSVC
3. SGDClassifier
4. BernoulliNB (BNB)
5. Decision Tree
6. Random Forest
7. Extra Trees
8. Bagging
9. LightGBM
10. XGBoost


In [ ]:
def get_base_model_builders(random_state=RANDOM_STATE):
    return OrderedDict({
        'LR': lambda: LogisticRegression(max_iter=2000, n_jobs=-1, class_weight='balanced', random_state=random_state),
        'LinearSVC': lambda: LinearSVC(dual=False, class_weight='balanced', random_state=random_state),
        'SGD': lambda: SGDClassifier(loss='log_loss', max_iter=2000, class_weight='balanced', random_state=random_state),
        'BNB': lambda: BernoulliNB(),
        'DT': lambda: DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
        'RF': lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', n_jobs=-1, random_state=random_state),
        'ET': lambda: ExtraTreesClassifier(n_estimators=300, class_weight='balanced_subsample', n_jobs=-1, random_state=random_state),
        'Bagging': lambda: BaggingClassifier(
            estimator=DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
            n_estimators=120,
            n_jobs=-1,
            random_state=random_state,
        ),
        'LGBM': lambda: lgb.LGBMClassifier(
            objective='multiclass',
            num_class=3,
            n_estimators=300,
            random_state=random_state,
            class_weight='balanced',
        ),
        'XGB': lambda: xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=3,
            n_estimators=300,
            eval_metric='mlogloss',
            random_state=random_state,
            n_jobs=-1,
            tree_method='hist',
        ),
    })


def calculate_gmean(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    recalls = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    recalls = np.where(recalls == 0, 1e-12, recalls)
    return float(np.prod(recalls) ** (1 / len(recalls)))


def comprehensive_evaluate(y_true, y_pred):
    return {
        'Weighted F1-score': f1_score(y_true, y_pred, average='weighted') * 100,
        'Macro F1-score': f1_score(y_true, y_pred, average='macro') * 100,
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa Score': cohen_kappa_score(y_true, y_pred),
        'G-Mean': calculate_gmean(y_true, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
    }


BASE_MODEL_BUILDERS = get_base_model_builders()
print('Final model set:', list(BASE_MODEL_BUILDERS.keys()))


## 10. Train and Evaluate Base Models (100,000 per Domain, 80/20 split)

This is the main experiment block for the 10 fixed models.


In [ ]:
def train_domain_base_models(domain_name, df, model_builders):
    print('\\n' + '=' * 80)
    print(f'Training base models for domain: {domain_name}')
    print('=' * 80)

    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'],
        df['label'],
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=df['label'],
    )

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=100_000,
        min_df=5,
        sublinear_tf=True,
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    trained_models = {}
    predictions = {}
    metrics_rows = []

    for model_name, builder in model_builders.items():
        print(f'Training {model_name}...')
        model = builder()
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)

        trained_models[model_name] = model
        predictions[model_name] = y_pred

        row = {'Model': model_name}
        row.update(comprehensive_evaluate(y_test, y_pred))
        metrics_rows.append(row)

    metrics_df = pd.DataFrame(metrics_rows).sort_values('Weighted F1-score', ascending=False).reset_index(drop=True)

    return {
        'domain': domain_name,
        'df': df,
        'X_train_text': X_train,
        'X_test_text': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'vectorizer': vectorizer,
        'X_train_vec': X_train_vec,
        'X_test_vec': X_test_vec,
        'models': trained_models,
        'predictions': predictions,
        'metrics_df': metrics_df,
    }


domain_runs = {}
for domain_name, df in domain_dfs.items():
    domain_runs[domain_name] = train_domain_base_models(domain_name, df, BASE_MODEL_BUILDERS)


## 11. Base Model Results Tables

In [ ]:
for domain_name, run in domain_runs.items():
    print('\\n' + '=' * 100)
    print(f'{domain_name} - Base Model Metrics')
    print('=' * 100)
    display(run['metrics_df'])


## 12. StackFull Ensemble Training (10 Base Models, CV=5)

StackFull uses all 10 base models with logistic regression as the meta-learner.


In [ ]:
def get_stacking_estimators(model_names):
    estimators = []
    for name in model_names:
        base_estimator = BASE_MODEL_BUILDERS[name]()

        # Stacking with probability meta-features requires predict_proba.
        if name == 'LinearSVC':
            base_estimator = CalibratedClassifierCV(base_estimator, method='sigmoid', cv=3)

        estimators.append((name, base_estimator))
    return estimators


def train_stack_model(X_train_vec, y_train, estimator_names):
    estimators = get_stacking_estimators(estimator_names)

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        stack_method='predict_proba',
        cv=5,
        n_jobs=-1,
        passthrough=False,
    )

    stack.fit(X_train_vec, y_train)
    return stack


for domain_name, run in domain_runs.items():
    print(f'Training StackFull for {domain_name}...')
    stack_full = train_stack_model(run['X_train_vec'], run['y_train'], list(BASE_MODEL_BUILDERS.keys()))
    y_pred_stack = stack_full.predict(run['X_test_vec'])

    run['stack_full_model'] = stack_full
    run['predictions']['StackFull'] = y_pred_stack
    run['stack_full_metrics'] = comprehensive_evaluate(run['y_test'], y_pred_stack)

    print(f"{domain_name} StackFull Weighted F1: {run['stack_full_metrics']['Weighted F1-score']:.3f}")


## 13. SHAP Model-Level Ranking on StackFull Meta-Features

This section performs the paper's key contribution:
- SHAP on meta-learner inputs (base-model probability features)
- mean absolute SHAP per base model
- threshold rule (> 0.03) and top-6 selection for StackSHAP
- Figure-style model importance bars (overall + per-class)


In [ ]:
def to_shap_array(shap_values_obj):
    if isinstance(shap_values_obj, list):
        arr = np.stack(shap_values_obj, axis=2)
        return arr

    if hasattr(shap_values_obj, 'values'):
        arr = np.array(shap_values_obj.values)
    else:
        arr = np.array(shap_values_obj)

    if arr.ndim == 2:
        arr = arr[:, :, None]
    return arr


def model_level_shap_analysis(run, domain_name, threshold=0.03):
    stack_model = run['stack_full_model']
    X_train_vec = run['X_train_vec']
    X_test_vec = run['X_test_vec']

    # Meta-features produced by StackFull (base model probabilities)
    meta_train = stack_model.transform(X_train_vec)
    meta_test = stack_model.transform(X_test_vec)

    base_names = [name for name, _ in stack_model.estimators]
    classes_ = list(stack_model.classes_)
    n_classes = len(classes_)

    meta_feature_names = []
    for model_name in base_names:
        for cls in classes_:
            meta_feature_names.append(f'{model_name}_prob_{cls}')

    explainer = shap.LinearExplainer(
        stack_model.final_estimator_,
        meta_train,
        feature_perturbation='interventional',
    )

    shap_values_obj = explainer.shap_values(meta_test)
    shap_arr = to_shap_array(shap_values_obj)

    # Handle shape inconsistencies defensively
    if shap_arr.shape[1] != meta_test.shape[1] and shap_arr.shape[0] == meta_test.shape[1]:
        shap_arr = np.transpose(shap_arr, (1, 0, 2))

    # Overall feature importance
    feature_importance = np.mean(np.abs(shap_arr), axis=(0, 2))

    # Aggregate feature importance to model importance
    model_importance = {}
    per_class_importance = {int(cls): {} for cls in classes_}

    for i, model_name in enumerate(base_names):
        idx_start = i * n_classes
        idx_end = (i + 1) * n_classes

        model_importance[model_name] = float(np.mean(feature_importance[idx_start:idx_end]))

        for class_idx, cls in enumerate(classes_):
            class_vals = np.mean(np.abs(shap_arr[:, idx_start:idx_end, class_idx]), axis=(0, 1))
            per_class_importance[int(cls)][model_name] = float(class_vals)

    sorted_models = sorted(model_importance.items(), key=lambda x: x[1], reverse=True)

    selected = [m for m, imp in sorted_models if imp > threshold]
    if len(selected) < 6:
        selected = [m for m, _ in sorted_models[:6]]
    else:
        selected = selected[:6]

    print(f'\n{domain_name} model-level SHAP ranking:')
    for m, imp in sorted_models:
        print(f'  {m:<10} {imp:.5f}')

    print(f'{domain_name} StackSHAP selected models (threshold>{threshold}): {selected}')

    # Fig. 10 style: overall aggregated model importance
    plt.figure(figsize=(10, 6))
    labels = [m for m, _ in sorted_models]
    values = [v for _, v in sorted_models]
    sns.barplot(x=values, y=labels, palette='viridis')
    plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold={threshold}')
    plt.title(f'{domain_name}: Aggregated Base Model Importance (SHAP)')
    plt.xlabel('Mean |SHAP| Importance')
    plt.ylabel('Base Model')
    plt.legend()
    plt.show()

    # Fig. 11 style: per-class model importance
    per_class_df = pd.DataFrame(per_class_importance).T
    per_class_df.index = [f'Class {i}' for i in per_class_df.index]

    plt.figure(figsize=(12, 6))
    for model_name in labels:
        plt.plot(per_class_df.index, per_class_df[model_name], marker='o', label=model_name)
    plt.axhline(threshold, color='red', linestyle='--', label=f'Threshold={threshold}')
    plt.title(f'{domain_name}: Per-Class Base Model Importance (SHAP)')
    plt.ylabel('Mean |SHAP| Importance')
    plt.xlabel('Class')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    run['stack_meta_train'] = meta_train
    run['stack_meta_test'] = meta_test
    run['stack_meta_feature_names'] = meta_feature_names
    run['stack_model_importance'] = model_importance
    run['stack_model_importance_sorted'] = sorted_models
    run['stack_model_per_class_importance'] = per_class_importance
    run['stackshap_selected_models'] = selected


for domain_name, run in domain_runs.items():
    model_level_shap_analysis(run, domain_name, threshold=0.03)


## 14. Train StackSHAP (Top-6 Models from SHAP Ranking)

StackSHAP is built from the selected top-6 models per domain.


In [ ]:
for domain_name, run in domain_runs.items():
    selected_models = run['stackshap_selected_models']
    print(f'Training StackSHAP for {domain_name} using: {selected_models}')

    stack_shap = train_stack_model(run['X_train_vec'], run['y_train'], selected_models)
    y_pred_stack_shap = stack_shap.predict(run['X_test_vec'])

    run['stack_shap_model'] = stack_shap
    run['predictions']['StackSHAP'] = y_pred_stack_shap
    run['stack_shap_metrics'] = comprehensive_evaluate(run['y_test'], y_pred_stack_shap)

    print(f"{domain_name} StackSHAP Weighted F1: {run['stack_shap_metrics']['Weighted F1-score']:.3f}")


## 15. Final Performance Tables (Base + StackFull + StackSHAP)

Tables are generated in paper-style metric format for all domains.


In [ ]:
def build_final_table(run):
    base_df = run['metrics_df'].copy()

    stack_full_row = {'Model': 'StackFull'}
    stack_full_row.update(run['stack_full_metrics'])

    stack_shap_row = {'Model': 'StackSHAP'}
    stack_shap_row.update(run['stack_shap_metrics'])

    final_df = pd.concat([
        base_df,
        pd.DataFrame([stack_full_row, stack_shap_row]),
    ], ignore_index=True)

    final_df = final_df.sort_values('Weighted F1-score', ascending=False).reset_index(drop=True)
    return final_df


final_tables = {}
for domain_name, run in domain_runs.items():
    table = build_final_table(run)
    final_tables[domain_name] = table

    print('\\n' + '=' * 110)
    print(f'{domain_name} - Final Model Comparison Table')
    print('=' * 110)
    display(table)


## 16. Normalized Confusion Matrices (StackFull vs StackSHAP)

Percentages are normalized by true class (row-wise), matching the paper style.


In [ ]:
def plot_normalized_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2], normalize='true') * 100
    sns.heatmap(
        cm,
        annot=True,
        fmt='.1f',
        cmap='Blues',
        cbar=False,
        xticklabels=['Negative', 'Neutral', 'Positive'],
        yticklabels=['Negative', 'Neutral', 'Positive'],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')


for domain_name, run in domain_runs.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_normalized_cm(run['y_test'], run['predictions']['StackFull'], f'{domain_name} - StackFull (%)', axes[0])
    plot_normalized_cm(run['y_test'], run['predictions']['StackSHAP'], f'{domain_name} - StackSHAP (%)', axes[1])
    plt.tight_layout()
    plt.show()


## 17. Statistical Validation with McNemar's Test (p < 0.05)

Per domain we compute:
1. StackFull vs each of 10 base models
2. StackSHAP vs each of its selected base models
3. StackFull vs StackSHAP


In [ ]:
def mcnemar_pvalue(y_true, pred_a, pred_b):
    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)

    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    table = np.array([
        [np.sum(correct_a & correct_b), np.sum(correct_a & ~correct_b)],
        [np.sum(~correct_a & correct_b), np.sum(~correct_a & ~correct_b)],
    ])

    result = mcnemar(table, exact=False, correction=True)
    return float(result.pvalue), table


mcnemar_results = {}

for domain_name, run in domain_runs.items():
    y_true = run['y_test'].to_numpy()

    # (1) StackFull vs all base models
    c1_rows = []
    for model_name in BASE_MODEL_BUILDERS.keys():
        p, _ = mcnemar_pvalue(y_true, run['predictions']['StackFull'], run['predictions'][model_name])
        c1_rows.append({
            'Domain': domain_name,
            'Comparison': f'StackFull vs {model_name}',
            'p_value': p,
            'Significant(p<0.05)': p < 0.05,
        })

    # (2) StackSHAP vs selected base models
    c2_rows = []
    for model_name in run['stackshap_selected_models']:
        p, _ = mcnemar_pvalue(y_true, run['predictions']['StackSHAP'], run['predictions'][model_name])
        c2_rows.append({
            'Domain': domain_name,
            'Comparison': f'StackSHAP vs {model_name}',
            'p_value': p,
            'Significant(p<0.05)': p < 0.05,
        })

    # (3) StackFull vs StackSHAP
    p3, _ = mcnemar_pvalue(y_true, run['predictions']['StackFull'], run['predictions']['StackSHAP'])
    c3_rows = [{
        'Domain': domain_name,
        'Comparison': 'StackFull vs StackSHAP',
        'p_value': p3,
        'Significant(p<0.05)': p3 < 0.05,
    }]

    mcnemar_results[domain_name] = {
        'c1': pd.DataFrame(c1_rows),
        'c2': pd.DataFrame(c2_rows),
        'c3': pd.DataFrame(c3_rows),
    }


# Display tables
for domain_name in mcnemar_results:
    print('\\n' + '=' * 95)
    print(f'McNemar Results: {domain_name}')
    print('=' * 95)
    print('\\n(1) StackFull vs Base Models')
    display(mcnemar_results[domain_name]['c1'])
    print('\\n(2) StackSHAP vs Selected Base Models')
    display(mcnemar_results[domain_name]['c2'])
    print('\\n(3) StackFull vs StackSHAP')
    display(mcnemar_results[domain_name]['c3'])


# Plot p-values for paper-style interpretation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, domain_name in enumerate(domain_dfs.keys()):
    df_plot = mcnemar_results[domain_name]['c1']
    sns.barplot(x='Comparison', y='p_value', data=df_plot, ax=axes[i], palette='crest')
    axes[i].axhline(0.05, color='red', linestyle='--', label='p=0.05')
    axes[i].set_title(f'{domain_name}: StackFull vs Base')
    axes[i].tick_params(axis='x', rotation=90)
axes[0].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, domain_name in enumerate(domain_dfs.keys()):
    df_plot = mcnemar_results[domain_name]['c2']
    sns.barplot(x='Comparison', y='p_value', data=df_plot, ax=axes[i], palette='flare')
    axes[i].axhline(0.05, color='red', linestyle='--', label='p=0.05')
    axes[i].set_title(f'{domain_name}: StackSHAP vs Selected Base')
    axes[i].tick_params(axis='x', rotation=90)
axes[0].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
c3_all = pd.concat([mcnemar_results[d]['c3'] for d in mcnemar_results], ignore_index=True)
sns.barplot(x='Domain', y='p_value', data=c3_all, palette='magma')
plt.axhline(0.05, color='red', linestyle='--', label='p=0.05')
plt.title('StackFull vs StackSHAP: McNemar p-values by Domain')
plt.legend()
plt.show()


## 18. SHAP Feature-Level Interpretation for Bagging (Paper Representative)

Required outputs:
- SHAP beeswarm (Bagging classifier only), using 300 test instances
- SHAP waterfall plots for three example reviews (positive, neutral, negative)

Note:
- BaggingClassifier is explained via SHAP's model-function interface.
- To keep this tractable for 300 explanations, a dedicated Bagging explainability vectorizer is used.


In [ ]:
# Use Clothing as the representative domain for Bagging feature-level SHAP analysis
run = domain_runs['Clothing']

X_train_text = run['X_train_text']
X_test_text = run['X_test_text']
y_train = run['y_train'].to_numpy()
y_test = run['y_test'].to_numpy()

# Dedicated explainability vectorizer for Bagging SHAP runtime feasibility
bag_shap_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=800,
    min_df=5,
    sublinear_tf=True,
)

X_train_bag = bag_shap_vectorizer.fit_transform(X_train_text)
X_test_bag = bag_shap_vectorizer.transform(X_test_text)

bagging_explainer_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    n_estimators=120,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
bagging_explainer_model.fit(X_train_bag, y_train)

# 300 test instances for SHAP as requested
n_explain = min(300, X_test_bag.shape[0])
X_shap = X_test_bag[:n_explain].toarray()
y_shap = y_test[:n_explain]

# Background sample
bg_size = min(120, X_train_bag.shape[0])
X_bg = X_train_bag[:bg_size].toarray()

print('Computing SHAP values for Bagging model (this may take time)...')
explainer_bag = shap.Explainer(bagging_explainer_model.predict_proba, X_bg, feature_names=bag_shap_vectorizer.get_feature_names_out())
shap_values_bag = explainer_bag(X_shap, max_evals=401)

# Beeswarm for positive class (class 2)
shap.plots.beeswarm(shap_values_bag[:, :, 2], max_display=20)

# Waterfall plots for representative examples (predicted positive/neutral/negative)
pred_bag = bagging_explainer_model.predict(X_shap)

example_indices = {}
for target_class in [2, 1, 0]:
    idx_candidates = np.where(pred_bag == target_class)[0]
    if len(idx_candidates) == 0:
        idx_candidates = np.where(y_shap == target_class)[0]
    if len(idx_candidates) > 0:
        example_indices[target_class] = int(idx_candidates[0])

label_name = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for cls in [2, 1, 0]:
    if cls in example_indices:
        i = example_indices[cls]
        print(f'\nWaterfall explanation for {label_name[cls]} example (index={i})')
        shap.plots.waterfall(shap_values_bag[i, :, cls], max_display=15)


## 19. Extra Analysis (Kept): Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (domain_name, run) in zip(axes, domain_runs.items()):
    X = run['X_train_vec']
    y = run['y_train']

    estimator = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight='balanced', random_state=RANDOM_STATE)

    train_sizes, train_scores, val_scores = learning_curve(
        estimator,
        X,
        y,
        cv=5,
        scoring='f1_weighted',
        train_sizes=np.linspace(0.1, 1.0, 5),
        n_jobs=-1,
    )

    ax.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train')
    ax.plot(train_sizes, val_scores.mean(axis=1), marker='o', label='Validation')
    ax.set_title(f'Learning Curve: {domain_name}')
    ax.set_xlabel('Training Size')
    ax.set_ylabel('Weighted F1')
    ax.legend()

plt.tight_layout()
plt.show()


## 20. Extra Analysis (Kept): ROC/AUC Curves (One-vs-Rest)

In [ ]:
from sklearn.metrics import roc_curve, auc

class_names = ['Negative', 'Neutral', 'Positive']
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
auc_summary = []

for ax, (domain_name, run) in zip(axes, domain_runs.items()):
    lr_model = run['models']['LR']
    X_test_vec = run['X_test_vec']
    y_test = run['y_test'].to_numpy()

    y_score = lr_model.predict_proba(X_test_vec)
    y_bin = label_binarize(y_test, classes=[0, 1, 2])

    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        auc_val = auc(fpr, tpr)
        auc_summary.append((domain_name, class_name, auc_val))
        ax.plot(fpr, tpr, color=color, label=f'{class_name} (AUC={auc_val:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(f'ROC Curves (OvR): {domain_name}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

auc_df = pd.DataFrame(auc_summary, columns=['Domain', 'Class', 'AUC'])
print('AUC Summary Table')
display(auc_df.pivot(index='Class', columns='Domain', values='AUC'))


## 21. Run Completion Checklist

If all cells run successfully, this notebook produces:
- Metrics for all 10 base models + StackFull + StackSHAP in all domains
- Paper-style comparison tables
- Normalized confusion matrices (percentage-based)
- McNemar p-value comparisons and significance checks
- SHAP model-level ranking and threshold-based StackSHAP selection
- SHAP feature-level Bagging beeswarm and waterfall plots
- Rating/class distribution plots
- Learning curves and ROC/AUC (extra analyses kept)
